# Fine-tuning do modelo

Em `data-prep.ipynb`, usamos o modelo `gemma-4:e4b` para traduzir os pares de perguntas e respostas mais bem avaliados do MedQuAD e notamos que o modelo base muitas vezes gera respostas melhores do que as disponíveis no dataset. Por esse motivo, optamos por explorar outros modelos e identificamos o `llama3.2:3b` como ótimo candidato: é preparado para realizar tarefas (instruct), requer relativamente pouca memória para executar (2GB RAM), tem time to first token (ttft) muito melhor e parece ter menos conhecimento médico, tornando mais fácil avaliar o resultado do fine-tuning aqui proposto.

Começamos por carregar o dataset traduzido, então testamos o modelo antes do fine tuning, executamos o fine tuning e avaliamos as respostas geradas pelo modelo ajustado.


In [1]:

import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

!pip install langchain_ollama

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 19.8 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.8/110.8 MB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 48.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive

drive.mount('/content/drive')

CONTENT_FOLDER = '/content/drive/MyDrive/Colab Notebooks/data/fiap'

Mounted at /content/drive


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_ollama import ChatOllama
from langchain_core.runnables import RunnableConfig


MODEL = "llama3.2:3b"
CONCURRENCY = 4  # número de requests paralelos no Ollama
OUTPUT_CSV = "assets/MedQuAD_excellent_ptbr_respostas_antes.csv"
CONTEXT_LENGTH = 8192

# llm = ChatOllama(model=MODEL, temperature=0.1, num_ctx=CONTEXT_LENGTH)

assistant_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um assistente clínico de apoio a médicos no Brasil. "
            "Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. "
            "Evite inventar dados clínicos. "
            "Seja honesto sobre não saber a resposta. "
            "Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.",
        ),
        ("user", "{question}"),
        ("assistant", "{answer}")
    ]
)

# usamos o parser padrão aqui
# assistant_chain = assistant_prompt | llm | StrOutputParser()

In [19]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 8192 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/Llama-3.1-8B-bnb-4bit",      # Llama-3.1 2x faster
    "unsloth/Llama-3.1-8B-Instruct-bnb-4bit",
    "unsloth/Llama-3.1-70B-bnb-4bit",
    "unsloth/Llama-3.1-405B-bnb-4bit",    # 4bit for 405b!
    "unsloth/Mistral-Small-Instruct-2409",     # Mistral 22b 2x faster!
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit",
    "unsloth/Phi-3.5-mini-instruct",           # Phi-3.5 2x faster!
    "unsloth/Phi-3-medium-4k-instruct",
    "unsloth/gemma-2-9b-bnb-4bit",
    "unsloth/gemma-2-27b-bnb-4bit",            # Gemma 2x faster!

    "unsloth/Llama-3.2-1B-bnb-4bit",           # NEW! Llama 3.2 models
    "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
    "unsloth/Llama-3.2-3B-bnb-4bit",
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",

    "unsloth/Llama-3.3-70B-Instruct-bnb-4bit" # NEW! Llama 3.3 70B!
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


## Preparação do fine tuning



In [18]:
from datasets import load_dataset

dataset = load_dataset("csv", data_files=CONTENT_FOLDER + '/MedQuAD_excellent_ptbr.csv')

dataset

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['AnswerID', 'Answer', 'Question', 'qid', 'score', 'Pergunta_BR', 'Resposta_BR'],
        num_rows: 141
    })
})

In [20]:
def format_dataset_into_model_input(data):
    return {
        "text": assistant_prompt.format(question=data["Pergunta_BR"], answer=data["Resposta_BR"]) + tokenizer.eos_token
    }

dataset = dataset.map(format_dataset_into_model_input)

Map:   0%|          | 0/141 [00:00<?, ? examples/s]

In [21]:
dataset['train'][1]["text"]

'System: Você é um assistente clínico de apoio a médicos no Brasil. Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. Evite inventar dados clínicos. Seja honesto sobre não saber a resposta. Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.\nHuman: Quais são os tratamentos para Sífilis - primária? (Também chamada de: Sífilis primária; Sífilis secundária; Sífilis tardia; Sífilis terciária)\nAI: A sífilis pode ser tratada com antibióticos, tais como:\n- Doxiciclina\n- Penicilina G benzatina\n- Tetraciclina (para pacientes alérgicos à penicilina)\n\nA duração do tratamento depende da gravidade da sífilis e de fatores como a saúde geral do paciente. Para tratar a sífilis durante a gravidez, a penicilina é o medicamento de escolha. A tetraciclina não pode ser usada no tratamento porque é perigosa para o feto. A eritromicina pode não prevenir a sífilis congênita no 

In [22]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

In [51]:
from trl import SFTConfig, SFTTrainer
# from transformers import DataCollatorForSeq2Seq
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    # data_collator = DataCollatorForSeq2Seq(tokenizer = tokenizer),
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3, # Set this for 1 full training run.
        # max_steps = 60, # Use for fast training to test the setup and inference process
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use TrackIO/WandB etc
    ),
)

In [52]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 141 | Num Epochs = 3 | Total steps = 54
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,0.581100
2,0.523100
3,0.970300
4,0.541000
5,0.444900
6,0.429200
7,0.404000
8,0.334700
9,0.651100
10,0.853000


In [53]:
assistant_prompt_for_inference = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Você é um assistente clínico de apoio a médicos no Brasil. "
            "Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. "
            "Evite inventar dados clínicos. "
            "Seja honesto sobre não saber a resposta. "
            "Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.",
        ),
        ("user", "{question}"),
        ("assistant", "")
    ]
)

In [54]:
%%capture
FastLanguageModel.for_inference(model)

In [57]:
inputs = tokenizer(
[
    # assistant_prompt_for_inference.format(question=dataset['train'][10]["Pergunta_BR"])
    assistant_prompt_for_inference.format(question="Qual é a história do HPS para Hantavírus?")
], return_tensors = "pt").to("cuda")

print("Trained response:\n  ")
print(dataset['train'][10]["Resposta_BR"])

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)

print("Model response:\n")
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 1024, temperature=0.1)

# outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
# model_response = tokenizer.batch_decode(outputs)

Trained response:
  
Secreção de ouvido é o escoamento de sangue, cera de ouvido, pus ou fluido do ouvido.
Model response:

<|begin_of_text|>System: Você é um assistente clínico de apoio a médicos no Brasil. Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. Evite inventar dados clínicos. Seja honesto sobre não saber a resposta. Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.
Human: Qual é a história do HPS para Hantavírus?
AI:  Em maio de 1993, um caso de síndrome pulmonar indolor (SPI) entre adultos foi noticiado nos Estados Unidos. Este caso foi o início da investigação sobre a relação entre o hantavírus e a SPI. A doença foi descrita como uma condição potencialmente fatal que afeta principalmente os pulmões.<|eot_id|>


In [28]:
model_response

['<|begin_of_text|>System: Você é um assistente clínico de apoio a médicos no Brasil. Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. Evite inventar dados clínicos. Seja honesto sobre não saber a resposta. Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.\nHuman: Como diagnosticar sífilis primária? (Também chamado de: Sífilis primária; Sífilis secundária; Sífilis tardia; Sífilis terciária)\nAI: 1. **Exame físico:** O médico pode sentir a lesão na pele ou no tecido mucoso, como a língua, o nariz, o ouvido ou a vagina. 2. **Exame de sangue:** O médico coleta uma amostra de sangue para verificar se o paciente tem anticorpos contra a sífilis. 3. **Exame de líquido vaginal:** O médico coleta um líquido vaginal para verificar se o paciente tem anticorpos contra a sífilis. 4. **Exame de líquido ocular']

In [46]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
parser.invoke(model_response[0])

'<|begin_of_text|>System: Você é um assistente clínico de apoio a médicos no Brasil. Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. Evite inventar dados clínicos. Seja honesto sobre não saber a resposta. Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.\nHuman: Como diagnosticar sífilis primária? (Também chamado de: Sífilis primária; Sífilis secundária; Sífilis tardia; Sífilis terciária) AI: O diagnóstico de sífilis é feito por meio de exames de sangue para verificar a presença de anticorpos contra a sífilis. O exame de sangue é o método mais comum para diagnosticar a sífilis. O exame de sangue pode ser feito em qualquer momento do processo da doença. A sífilis é mais comum em homens de 20 a 40 anos de idade. O exame de sangue pode ser feito em qualquer parte do corpo, mas geralmente é realizado no braço. O médico fará uma pequ'

In [42]:
dataset['train'][0]["Resposta_BR"]

'O médico ou enfermeiro fará um exame físico. Os testes que podem ser realizados incluem: - Exame de fluido da lesão - Ecocardiograma, angiografia aórtica e cateterismo cardíaco para avaliar os grandes vasos sanguíneos e o coração - Punção lombar e exame do líquido cefalorraquidiano - Exames de sangue para rastreio de bactérias da sífilis (RPR, VDRL ou TRUST). Se os testes RPR, VDRL ou TRUST forem positivos, um dos seguintes testes será necessário para confirmar o diagnóstico: - FTA-ABS (teste de anticorpos treponêmicos fluorescentes) - MHA-TP - TP-EIA - TP-PA'

In [58]:
model.save_pretrained("/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model") # Local saving
tokenizer.save_pretrained("/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model")

('/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model/tokenizer_config.json',
 '/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model/special_tokens_map.json',
 '/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model/chat_template.jinja',
 '/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model/tokenizer.json')

In [ ]:
# Save to q4_k_m GGUF
if True: model.save_pretrained_gguf("/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model", tokenizer, quantization_method = "q4_k_m")

Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.46G [00:00<?, ?B/s]

Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [03:26<00:00, 103.34s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [07:26<00:00, 223.20s/it]


Unsloth: Merge process complete. Saved to `/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Updating system package directories
Unsloth: Cloning llama.cpp repository...
Unsloth: Building llama.cpp - please wait 1 to 3 minutes


In [4]:
%%capture
!pip install langchain_huggingface

In [5]:
from transformers import pipeline
from unsloth import FastLanguageModel
import torch

from langchain_huggingface import HuggingFacePipeline

# 1. Load the Unsloth Model
# max_seq_length = 2048
ft_model, ft_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/Colab Notebooks/data/fiap/fase3/lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = True,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(ft_model)

# 2. Create a Transformers Pipeline
pipe = pipeline(
    "text-generation",
    model=ft_model,
    tokenizer=ft_tokenizer,
    max_new_tokens=512,
    temperature=0.2,
)

# 3. Wrap into LangChain
ft_llm = HuggingFacePipeline(pipeline=pipe)

ft_chain = assistant_prompt_for_inference | ft_llm | StrOutputParser()


/usr/local/lib/python3.12/dist-packages/unsloth/__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [17]:
ft_chain.invoke({
    "question": "Como tratar sífilis?"
})

'System: Você é um assistente clínico de apoio a médicos no Brasil. Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. Evite inventar dados clínicos. Seja honesto sobre não saber a resposta. Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.\nHuman: Como tratar sífilis?\nAI: 1. **Também chamada de:** sífilis; sífilis primária; sífilis secundária; sífilis terciária; sífilis quaternária; sífilis congênita; sífilis neonatal; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis congênita; sífilis co

In [6]:
!pip install langchain_huggingface

In [59]:
from unsloth import FastLanguageModel
import torch

from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

# 1. Load the Unsloth Model
# max_seq_length = 2048
foundation_model, foundation_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Enable native 2x faster inference
FastLanguageModel.for_inference(foundation_model)

# 2. Create a Transformers Pipeline
pipe = pipeline(
    "text-generation",
    model=foundation_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.2,
)

# 3. Wrap into LangChain
foundation_llm = HuggingFacePipeline(pipeline=pipe)

foundation_chain = assistant_prompt_for_inference | foundation_llm | StrOutputParser()


==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Device set to use cuda:0


In [60]:
foundation_chain.invoke({
    "question": "Qual é a história do HPS para Hantavírus?"
})

'System: Você é um assistente clínico de apoio a médicos no Brasil. Recomende mas não prescreva medicamentos, doses ou esquemas terapêuticos específicos: o médico responsável decide. Evite inventar dados clínicos. Seja honesto sobre não saber a resposta. Responda em português do Brasil, de forma objetiva e profissional, apenas com informações relevantes à interação.\nHuman: Qual é a história do HPS para Hantavírus?\nAI:  O Hantavírus é um vírus que pode causar uma doença infecciosa conhecida como Hantavirus Pulmonary Syndrome (HPS). A história do Hantavírus remonta a 1993, quando foi identificado pela primeira vez na América do Norte, especificamente nos Estados Unidos.\n\nDesde então, o Hantavírus foi identificado em várias partes do mundo, incluindo a América do Sul, a Ásia e a Europa. A doença causada pelo Hantavírus é transmitida por meio de contato com fezes ou urina de roedores infectados, como a família dos roedores da água (Cricetidae).\n\nA doença HPS é caracterizada por sinto